1.comparaison entre images et leur corespondance (labels)

In [ ]:
from pathlib import Path

DATASET_DIR = Path("../trafic_data")

for split in ["train", "valid"]:
    images = list((DATASET_DIR / split / "images").glob("*"))
    labels = list((DATASET_DIR / split / "labels").glob("*.txt"))

    noms_images = {f.stem for f in images}   #f.stem récupère le nom du fichier sans extension.
    noms_labels = {f.stem for f in labels}

    print(f"\n{split.upper()} :")
    print("Images sans annotation :", len(noms_images - noms_labels))   #trouve les images sans Label correspondante
    print("Annotations sans image :", len(noms_labels - noms_images))   #trouve labels sans image correspondante


TRAIN :
Images sans annotation : 0
Annotations sans image : 0

VALID :
Images sans annotation : 0
Annotations sans image : 0


2.Examiner le format YOLO

In [ ]:
#afficher le nombre d'annotations vides pour chaque split
for split in ["train", "valid"]:
    labels = list((DATASET_DIR / split / "labels").glob("*.txt"))
    vides = [f for f in labels if f.stat().st_size == 0]

    print(f"{split.upper()} : {len(vides)} annotations vides")

TRAIN : 2 annotations vides
VALID : 0 annotations vides


In [18]:
#Afficher la structure du fichier text d'annotation
label_path = next(
    fichier for fichier in (DATASET_DIR / "train" / "labels").glob("*.txt")
    if fichier.stat().st_size > 0    
)

print("Fichier :", label_path.name)
print(label_path.read_text())

Fichier : 02_jpg.rf.65a084066fc353cd023eb5c953f40efe.txt
17 0.9546875 0.6892523364485982 0.0859375 0.11682242990654206
19 0.721875 0.5607476635514018 0.0953125 0.11448598130841121
17 0.3671875 0.6635514018691588 0.05625 0.10514018691588785
10 0.2828125 0.48130841121495327 0.0171875 0.037383177570093455
10 0.4015625 0.5327102803738317 0.01875 0.04906542056074766
10 0.675 0.4766355140186916 0.0234375 0.0397196261682243
10 0.696875 0.4602803738317757 0.0171875 0.030373831775700934
10 0.3390625 0.3317757009345794 0.0078125 0.014018691588785047
10 0.4015625 0.308411214953271 0.00625 0.009345794392523364
10 0.478125 0.3317757009345794 0.0078125 0.016355140186915886
16 0.2984375 0.3107476635514019 0.0140625 0.016355140186915886
11 0.315625 0.308411214953271 0.0140625 0.02336448598130841
4 0.2828125 0.29906542056074764 0.01875 0.037383177570093455
4 0.2796875 0.2733644859813084 0.0171875 0.014018691588785047
17 0.3296875 0.4369158878504673 0.021875 0.04672897196261682
17 0.30625 0.380841121495

In [19]:
valeurs = label_path.read_text().splitlines()[0].split()

print("Value numbers :", len(valeurs))
print("Class ID :", valeurs[0])
print("X center  :", valeurs[1])
print("Y center :", valeurs[2])
print("width :", valeurs[3])
print("height :", valeurs[4])

Value numbers : 5
Class ID : 17
X center  : 0.9546875
Y center : 0.6892523364485982
width : 0.0859375
height : 0.11682242990654206


In [ ]:
#vérifie la normalisation des coordonnées et des dimensions des bounding boxes au format YOLO.
x, y, largeur, hauteur = map(float, valeurs[1:]) # Convertir les valeurs en nombres

print("Centre X valide :", 0 <= x <= 1)
print("Centre Y valide :", 0 <= y <= 1)
print("Largeur valide :", 0 < largeur <= 1)
print("Hauteur valide :", 0 < hauteur <= 1)

0.9546875
Centre X valide : True
Centre Y valide : True
Largeur valide : True
Hauteur valide : True


In [24]:
#Vérifier le fichier data.yaml, les noms des classes et les identifiants utilisés dans les annotations
for split in ["train", "valid"]:
    class_ids = set()

    for label_file in (DATASET_DIR / split / "labels").glob("*.txt"):
        for ligne in label_file.read_text().splitlines():
            if ligne.strip():
                class_ids.add(int(ligne.split()[0]))

    invalides = sorted(cid for cid in class_ids if cid < 0 or cid >= 21)

    print(f"\n{split.upper()}")
    print("Identifiants utilisés :", sorted(class_ids))
    print("Identifiants invalides :", invalides)


TRAIN
Identifiants utilisés : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Identifiants invalides : []

VALID
Identifiants utilisés : [3, 4, 5, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Identifiants invalides : []


In [26]:
# Vérifier les images corrompues
from PIL import Image

# Vérifier les images corrompues dans train et valid
for split in ["train", "valid"]:
    images_dir = DATASET_DIR / split / "images"
    images_corrompues = []

    for image_path in images_dir.iterdir():
        if image_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]:
            try:
                with Image.open(image_path) as img:
                    img.verify()
            except Exception:
                images_corrompues.append(image_path.name)

    print(f"\n{split.upper()}")
    print("Nombre d'images corrompues :", len(images_corrompues))
    print("Fichiers :", images_corrompues)


TRAIN
Nombre d'images corrompues : 0
Fichiers : []

VALID
Nombre d'images corrompues : 0
Fichiers : []


In [27]:
# Identifier les lignes invalides
for split in ["train", "valid"]:
    labels_dir = DATASET_DIR / split / "labels"
    lignes_invalides = []

    for label_file in labels_dir.glob("*.txt"):
        for numero, ligne in enumerate(
            label_file.read_text(encoding="utf-8").splitlines(), start=1
        ):
            if not ligne.strip():
                continue

            try:
                valeurs = ligne.split()

                if len(valeurs) != 5:
                    raise ValueError("Nombre de valeurs différent de 5")

                class_id = int(valeurs[0])
                x, y, largeur, hauteur = map(float, valeurs[1:])

                if not (0 <= class_id < 21):
                    raise ValueError("Identifiant de classe invalide")

                if not (
                    0 <= x <= 1 and
                    0 <= y <= 1 and
                    0 < largeur <= 1 and
                    0 < hauteur <= 1
                ):
                    raise ValueError("Coordonnées ou dimensions invalides")

            except (ValueError, OverflowError):
                lignes_invalides.append(
                    f"{label_file.name}, ligne {numero}: {ligne}"
                )

    print(f"\n{split.upper()}")
    print("Nombre de lignes invalides :", len(lignes_invalides))
    print("Détails :", lignes_invalides[:10])


TRAIN
Nombre de lignes invalides : 0
Détails : []

VALID
Nombre de lignes invalides : 24
Détails : ['Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 1: 9 0 0 0 0', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 2: 15 0 0 0 0', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 3: 13 0 0 0 0', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 4: 13 0 0 0 0', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 5: 13 0 0 0 0', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 6: 13 0 0 0 0', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 7: 13 0 0 0 0', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 8: 13 0 0 0 0', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 9: 13 0 0 0 0', 'Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt, ligne 10: 13 0 0 0 0']


In [ ]:
# Identifier les fichiers d'annotations vides
for split in ["train", "valid"]:
    labels_dir = DATASET_DIR / split / "labels"

    annotations_vides = [
        fichier.name
        for fichier in labels_dir.glob("*.txt")
        if not fichier.read_text(encoding="utf-8").strip()
    ]

    print(f"\n{split.upper()}")
    print("Nombre d'annotations vides :", len(annotations_vides))
    print("Fichiers :", annotations_vides)


TRAIN
Nombre d'annotations vides : 2
Fichiers : ['01_jpg.rf.8d8a2f0f90d5b83893cd252acd832c93.txt', '231_jpg.rf.42292592b7fed42abf94d9624eda3fa5.txt']

VALID
Nombre d'annotations vides : 0
Fichiers : []


In [ ]:
#check where we have 24lignes Invalides (Validation)
from collections import Counter

labels_dir = DATASET_DIR / "valid" / "labels"
lignes_invalides_par_fichier = Counter()

for label_file in labels_dir.glob("*.txt"):
    for ligne in label_file.read_text(encoding="utf-8").splitlines():
        if not ligne.strip():
            continue

        valeurs = ligne.split()

        try:
            if len(valeurs) != 5:
                raise ValueError

            class_id = int(valeurs[0])
            x, y, largeur, hauteur = map(float, valeurs[1:])

            if not (
                0 <= class_id < 21
                and 0 <= x <= 1
                and 0 <= y <= 1
                and 0 < largeur <= 1
                and 0 < hauteur <= 1
            ):
                lignes_invalides_par_fichier[label_file.name] += 1

        except (ValueError, OverflowError):
            lignes_invalides_par_fichier[label_file.name] += 1

print("Fichiers contenant des lignes invalides :")
for nom, nombre in lignes_invalides_par_fichier.items():
    print(f"{nom} : {nombre} lignes")

Fichiers contenant des lignes invalides :
Pias--359-_PNG.rf.43bcf36efe5cf8c37552d2c45fffea60.txt : 14 lignes
Pias--360-_PNG.rf.8405b0e44009a9300e0a1100ccf7d5b3.txt : 10 lignes
